# UCF-Crime → MTFL 이상 탐지 학습 파이프라인

이 노트북은 **UCF-Crime** 데이터셋(프레임 이미지 + 시간 구간 annotation)을 **MTFL(Multi-scale Temporal Feature Learning)** 이상 탐지 프레임워크에 맞게 전처리·학습·평가하는 전 과정을 다룹니다.

## 전체 흐름

1. **이미지 → MP4 변환** — 카테고리별 PNG 프레임을 영상으로 묶고, 원본 30fps 기준 anomaly 구간을 출력 fps에 맞게 재인덱싱
2. **train/test annotation 생성** — `Temporal_Anomaly_Annotation.txt`와 변환 결과를 합쳐 MTFL 형식의 `train_anno.txt`, `test_anno.txt` 작성
3. **MTFL feature 추출** — Swin3D 백본으로 L8/L32/L64 멀티스케일 특징 저장
4. **공통 feature만 필터링** — 세 스케일 모두 추출된 영상만 남기고 annotation 동기화
5. **Detection 학습·테스트·시각화** — MTFL detection 모델 학습, AUC/AP 로그 확인, 점수·GT·프레임 시각화

## 사전 준비

- `ucf_utils.py`, `MTFL_vis.py`가 **이 노트북과 같은 폴더**(`notebooks/동원/`)에 있어야 합니다.
- 아래 셀의 `FRAME_ROOT`, `ANNOTATION_TXT`, `OUT_ROOT`, `MTFL_ROOT` 등 경로를 **본인 환경**에 맞게 수정하세요.
- MTFL 저장소, Swin3D 사전학습 가중치, ffmpeg가 설치되어 있어야 feature 추출·영상 인코딩이 동작합니다.

---

## 1단계: 이미지 → MP4 변환 (설정)

In [ ]:
# ---------------------------------------------------------------------------
# 표준 라이브러리: 경로·랜덤 분할·(필요 시) 영상/JSON 처리
# cv2는 ucf_utils 내부에서 사용하며, 여기서는 직접 쓰이지 않을 수 있음
# ---------------------------------------------------------------------------
import os
import re
import cv2
import json
import random
from pathlib import Path
from collections import defaultdict

# 동일 폴더의 ucf_utils (프레임 수집·mp4 저장·구간 변환)
# import 오류 시: os.chdir/notebooks/동원) 또는 sys.path.insert(0, ".../동원")
from ucf_utils import (
    collect_frames_by_video,   # 카테고리 폴더 내 PNG를 video_id별로 그룹화
    frames_to_video,             # (frame_no, path) 리스트 → mp4
    timestamps_to_output_frame_intervals,  # JSON 초 단위 구간용 (txt 파이프라인에서는 미사용)
)

# =========================
# 경로 설정 (환경에 맞게 수정)
# =========================

# UCF-Crime **이미지 프레임** 루트 (클래스별 하위 폴더)
# 파일명 규칙: {video_id}_{원본프레임번호}.png  예) Abuse001_x264_0.png, _10.png ...
# 구조:
#   FRAME_ROOT/
#   ├── Abuse/
#   ├── Assault/
#   └── Normal/
FRAME_ROOT = Path("C:/4_1_딥러닝_팀플/UCF-Crime")

# UCF-Crime 공식 **시간 구간** annotation (한 줄 = 한 영상)
# 형식: {비디오파일명} {클래스} [시작프레임 끝프레임] ...  (-1,-1은 구간 없음)
ANNOTATION_TXT = Path(
    "C:/4_1_딥러닝_팀플/UCF-Crime-Annotations/Temporal_Anomaly_Annotation.txt"
)

# MTFL 파이프라인용 **커스텀 데이터셋** 루트 (변환 mp4 + annotation + features 저장)
OUT_ROOT = Path("C:/4_1_딥러닝_팀플/MTFL_UCF_custom")

VIDEO_OUT = OUT_ROOT / "videos"       # {카테고리}/{video_id}.mp4
ANNO_OUT = OUT_ROOT / "annotations"   # train_anno.txt, test_anno.txt 등

VIDEO_OUT.mkdir(parents=True, exist_ok=True)
ANNO_OUT.mkdir(parents=True, exist_ok=True)

# --- fps / 프레임 간격 ---
# UCF-Crime 원본은 30fps 가정. PNG가 10프레임마다 1장이면 실질 샘플링 간격 = 10
ORIGINAL_FPS = 30
DEFAULT_FRAME_STEP = 10

# mp4에 기록할 fps: 샘플링된 프레임만 이어 붙이므로, 재생 시간을 맞추려면
#   OUTPUT_FPS = ORIGINAL_FPS / DEFAULT_FRAME_STEP  →  3 fps
# 이후 annotation의 프레임 인덱스도 DEFAULT_FRAME_STEP으로 나누어 mp4 프레임에 맞춤
OUTPUT_FPS = ORIGINAL_FPS / DEFAULT_FRAME_STEP

# train/test 분할 재현성 (아래 셀들에서 동일 seed 사용)
random.seed(42)

ModuleNotFoundError: No module named 'ucf_utils'

## 1단계 (계속): 전체 카테고리 순회 — MP4 생성 + `all_items` 수집

- `Temporal_Anomaly_Annotation.txt`를 파싱해 **video_id → anomaly 구간(원본 30fps 프레임 인덱스)** 맵 생성
- `FRAME_ROOT`의 각 클래스 폴더에서 프레임을 모아 mp4로 저장 (이미 있으면 스킵)
- test용 annotation에는 **구간 + total_frames**가 필요하므로 `intervals`를 출력 fps 기준으로 변환해 `all_items`에 적재

In [ ]:
# 이후 train/test 분할·feature 필터링에 쓰일 메타데이터 리스트
all_items = []

# =========================
# 1) txt annotation 로드 → anno_map[video_id]
# =========================
# 키: video_id (확장자 없음, 예: Abuse028_x264)
# 값: category(라벨), intervals_raw[(s,e), ...] — s,e는 **원본 30fps** 프레임 인덱스
anno_map = {}

with open(ANNOTATION_TXT, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        parts = line.split()
        video_name = parts[0]          # 예: Abuse028_x264.mp4
        category = parts[1]            # 예: Abuse, Normal, Shoplifting ...
        nums = list(map(int, parts[2:]))  # (start, end) 쌍이 이어짐; -1,-1은 해당 구간 없음

        video_id = Path(video_name).stem  # mp4 확장자 제거 → 프레임/출력 파일명과 동일 id

        intervals_raw = []
        for i in range(0, len(nums), 2):
            s, e = nums[i], nums[i + 1]
            if s != -1 and e != -1:
                intervals_raw.append((s, e))

        anno_map[video_id] = {
            "category": category,
            "intervals_raw": intervals_raw,
        }

print("Loaded txt annotations:", len(anno_map))


# =========================
# 2) frame 폴더 순회 → mp4 변환 + all_items 적재
# =========================
for category_dir in FRAME_ROOT.iterdir():
    if not category_dir.is_dir():
        continue

    category = category_dir.name  # 폴더명 = 대략적인 클래스 (Normal 폴더 등)
    print(f"\nProcessing category: {category}")

    # 같은 video_id끼리 (frame_no, png_path) 정렬된 리스트로 묶음
    grouped = collect_frames_by_video(category_dir)
    print(f"  videos found: {len(grouped)}")

    for video_id, frame_items in grouped.items():
        # 프레임 수가 너무 적으면 MTFL feature/학습에 부적합 → 스킵
        if len(frame_items) < 8:
            print("  skip too short:", video_id, len(frame_items))
            continue

        out_video_path = VIDEO_OUT / category / f"{video_id}.mp4"

        # PNG 시퀀스 → mp4 (이미 변환된 파일이 있으면 재인코딩 생략)
        if not out_video_path.exists():
            frames_to_video(frame_items, out_video_path, OUTPUT_FPS)

        # 출력 mp4의 프레임 수 = 샘플링된 PNG 개수 (10프레임 간격이면 len(frame_items))
        total_output_frames = len(frame_items)

        # 공식 txt에서 이 video_id의 anomaly 구간 조회 (없으면 빈 리스트)
        raw_intervals = anno_map.get(video_id, {}).get("intervals_raw", [])

        intervals = []
        for s, e in raw_intervals:
            # 원본 30fps 인덱스 → 출력 mp4 인덱스 (DEFAULT_FRAME_STEP으로 다운샘플)
            out_s = int(s / DEFAULT_FRAME_STEP)
            out_e = int(e / DEFAULT_FRAME_STEP)

            # 클리핑: [0, total_output_frames - 1]
            out_s = max(0, min(out_s, total_output_frames - 1))
            out_e = max(0, min(out_e, total_output_frames - 1))

            # 유효 구간만 저장 (시작 < 끝)
            if out_e > out_s:
                intervals.append((out_s, out_e))

        # MTFL annotation에서 쓰는 상대 경로 (videos/ 기준)
        rel_video = f"{category}/{video_id}.mp4"

        # 라벨: txt에 있으면 공식 category, 없으면 Normal 폴더면 Normal, 아니면 폴더명
        if video_id in anno_map:
            label = anno_map[video_id]["category"]
        else:
            label = "Normal" if category.lower() == "normal" else category

        all_items.append({
            "rel_video": rel_video,           # train/test txt 첫 컬럼
            "label": label,                   # 두 번째 컬럼
            "total_frames": total_output_frames,  # test txt에 필요
            "intervals": intervals,           # test txt: start end 쌍 (출력 fps 기준)
        })

print("\nTotal converted videos:", len(all_items))

In [ ]:
# ---------------------------------------------------------------------------
# 2단계: train / test annotation 파일 생성 (MTFL 형식)
# ---------------------------------------------------------------------------
# train 한 줄:  {rel_video} {label}
# test 한 줄:   {rel_video} {label} {total_frames} [s1 e1 s2 e2 ...]
#   - test에만 frame-level GT 구간이 필요 (detection 평가용)
# ---------------------------------------------------------------------------
import random
from pathlib import Path

random.seed(42)
random.shuffle(all_items)  # 셔플 후 앞 20%를 test로 사용 (고정 seed)

test_ratio = 0.2
n_test = int(len(all_items) * test_ratio)

test_items = all_items[:n_test]
train_items = all_items[n_test:]

train_anno_path = ANNO_OUT / "train_anno.txt"
test_anno_path = ANNO_OUT / "test_anno.txt"

with open(train_anno_path, "w", encoding="utf-8") as f:
    for item in train_items:
        f.write(f"{item['rel_video']} {item['label']}\n")

with open(test_anno_path, "w", encoding="utf-8") as f:
    for item in test_items:
        line = f"{item['rel_video']} {item['label']} {item['total_frames']}"

        for s, e in item["intervals"]:
            line += f" {s} {e}"

        f.write(line + "\n")

print("train:", len(train_items), train_anno_path)
print("test:", len(test_items), test_anno_path)

In [ ]:
# 생성된 annotation 상위 10줄 미리보기 (형식·라벨·구간 확인용)
print("===== train sample =====")
print("\n".join(train_anno_path.read_text(encoding="utf-8").splitlines()[:10]))

print("\n===== test sample =====")
print("\n".join(test_anno_path.read_text(encoding="utf-8").splitlines()[:10]))

## 2단계 (중복 실행용): train/test annotation 다시 생성

아래 셀은 **위 셀(4~5)과 동일한 로직**입니다. `all_items`만 메모리에 남아 있으면 annotation 파일만 다시 쓸 때 사용합니다.

In [ ]:
# ---------------------------------------------------------------------------
# [중복] train/test annotation 재생성 — 셀 4와 동일 로직
# 주의: random.seed(42) 없이 shuffle → 셀 4와 train/test 구성이 달라질 수 있음
# ---------------------------------------------------------------------------
random.shuffle(all_items)

test_ratio = 0.2
n_test = int(len(all_items) * test_ratio)

test_items = all_items[:n_test]
train_items = all_items[n_test:]

train_anno_path = ANNO_OUT / "train_anno.txt"
test_anno_path = ANNO_OUT / "test_anno.txt"

with open(train_anno_path, "w", encoding="utf-8") as f:
    for item in train_items:
        f.write(f"{item['rel_video']} {item['label']}\n")

with open(test_anno_path, "w", encoding="utf-8") as f:
    for item in test_items:
        line = f"{item['rel_video']} {item['label']} {item['total_frames']}"

        for s, e in item["intervals"]:
            line += f" {s} {e}"

        f.write(line + "\n")

print("train:", len(train_items), train_anno_path)
print("test:", len(test_items), test_anno_path)

## 2단계 (계속): annotation 샘플 및 클래스 분포 확인

- train/test 파일 앞 5줄 출력
- `print_stats`: Normal vs Abnormal 비율, 클래스별 개수 (데이터 불균형 점검)

In [ ]:
# 파일 전체가 아닌 앞 5줄만 리스트로 출력
print("===== train_anno sample =====")
print(train_anno_path.read_text(encoding="utf-8").splitlines()[:5])

print("\n===== test_anno sample =====")
print(test_anno_path.read_text(encoding="utf-8").splitlines()[:5])

In [ ]:
from collections import Counter


def print_stats(items, name):
    """
    all_items에서 분할된 train_items / test_items의 라벨 통계 출력.
    Normal(대소문자 무시) vs 그 외(이상 행위 클래스) 비율을 확인합니다.
    """
    labels = [x["label"] for x in items]

    total = len(labels)

    normal_cnt = sum(1 for l in labels if l.lower() == "normal")
    abnormal_cnt = total - normal_cnt

    print(f"\n===== {name} =====")
    print(f"total videos     : {total}")
    print(f"normal videos    : {normal_cnt}")
    print(f"abnormal videos  : {abnormal_cnt}")

    print(f"normal ratio     : {normal_cnt / total:.4f}")
    print(f"abnormal ratio   : {abnormal_cnt / total:.4f}")

    print("\nlabel distribution:")
    counter = Counter(labels)

    for k, v in sorted(counter.items()):
        print(f"{k:15s}: {v}")


print_stats(train_items, "TRAIN")
print_stats(test_items, "TEST")

In [ ]:
# feature_extractor / OpenCV가 ffmpeg를 쓸 수 있는지 PATH 확인
import shutil

ffmpeg_path = shutil.which("ffmpeg")

print(ffmpeg_path)  # None이면 conda/pip로 ffmpeg 설치 후 경로 지정 필요

## 3단계: MTFL 멀티스케일 feature 추출 (Swin3D)

- `feature_extractor.py`가 `videos/` 아래 mp4를 읽어 **L8, L32, L64** 클립 길이별 `.txt` 특징 저장
- L8=짧은 클립(고해상도 시간), L64=긴 클립(넓은 temporal context) — detection에서 sf/mf/lf로 사용
- 사전학습 가중치: Kinetics-400 등으로 학습된 Swin3D `.pth`

In [ ]:
# conda 환경의 ffmpeg 바이너리 경로를 직접 지정해 인코더 목록 확인
# (PATH에 ffmpeg가 없을 때 feature 추출 전 점검용)
import subprocess
from pathlib import Path

# ★ 본인 PC의 ffmpeg.exe 경로로 수정
ffmpeg_path = Path(
    r"C:\Users\dongwon\anaconda3\pkgs\ffmpeg-8.1.1-gpl_h7d7abef_901\Library\bin\ffmpeg.exe"
)

result = subprocess.run(
    [str(ffmpeg_path), "-hide_banner", "-encoders"],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

print("return:", result.returncode)
print("libx264 있음?:", "libx264" in result.stdout)  # mp4 인코딩에 필요할 수 있음
print(result.stderr[-1000:])

## 3단계 (실행): feature_extractor 호출

**반드시 수정할 경로**

| 변수 | 설명 |
|------|------|
| `os.chdir(...)` | MTFL 저장소 루트로 작업 디렉터리 변경 (상대 import·config용) |
| `MTFL_ROOT` | MTFL 클론 경로 |
| `CUSTOM_ROOT` | 1단계에서 만든 `MTFL_UCF_custom` (videos, features) |
| `weight_path` | Swin3D 사전학습 `.pth` |

`clip_length` 8 / 32 / 64를 순회하며 각각 `features/L8`, `L32`, `L64`에 저장합니다.

In [ ]:
import sys
import subprocess
from pathlib import Path
import os

# MTFL 레포 루트로 chdir — feature_extractor 내부 상대 경로·설정 파일 로딩용
os.chdir(r"C:\4_1_딥러닝_팀플\MTFL")

MTFL_ROOT = "C:/4_1_딥러닝_팀플/MTFL"  # MTFL GitHub 클론 경로
CUSTOM_ROOT = "C:/4_1_딥러닝_팀플/MTFL_UCF_custom"  # 1단계 OUT_ROOT와 동일하게 맞출 것

feature_script = f"{MTFL_ROOT}/utils/feature_extractor.py"
video_dir = f"{CUSTOM_ROOT}/videos"       # 입력: 클래스별 mp4
feature_dir = f"{CUSTOM_ROOT}/features"   # 출력: L8/, L32/, L64/ 하위에 .txt
weight_path = (
    "C:/4_1_딥러닝_팀플/MTFL_custom/"
    "swin_base_patch244_window877_kinetics400_22k.pth"
)

Path(feature_dir).mkdir(parents=True, exist_ok=True)

# 영상이 짧으면 긴 clip_length(L64)에서 특징 추출이 실패할 수 있음 → 이후 common 필터링
for clip_length in [8, 32, 64]:
    print(f"\n========== Extracting L{clip_length} ==========")

    cmd = [
        sys.executable,
        feature_script,
        "--clip_length", str(clip_length),
        "--dataset_path", video_dir,
        "--save_dir", feature_dir,
        "--pretrained_3d", weight_path,
        "--gpu", "0",
        "--batch_size", "4",
        "--num_workers", "4",  # Windows에서 오류 시 0으로 낮추기
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    print("RETURN CODE:", result.returncode)
    print(result.stdout)
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"L{clip_length} failed")

## 3단계 (확인): 스케일별 추출된 feature 파일 개수

In [ ]:
from pathlib import Path

# 각 클립 길이 폴더 아래 모든 .txt feature 개수 (영상 1개 ≈ feature 1개 기대)
for L in ["L8", "L32", "L64"]:
    files = list(
        Path(f"C:/4_1_딥러닝_팀플/MTFL_UCF_custom/features/{L}").rglob("*.txt")
    )
    print(L, len(files))

## 4단계: L8 / L32 / L64 간 누락 영상 분석

짧은 영상은 L32·L64 슬라이딩 윈도우를 만들 수 없어 feature가 생성되지 않습니다.  
L8 기준으로 L32·L64에 없는 `rel_path` 목록을 출력·저장합니다.

In [ ]:
from pathlib import Path

FEATURE_ROOT = Path("C:/4_1_딥러닝_팀플/MTFL_UCF_custom/features")

feature_sets = {}

for L in ["L8", "L32", "L64"]:
    files = list((FEATURE_ROOT / L).rglob("*.txt"))

    # 상대 경로 키: "Shoplifting/Shoplifting007_x264" (확장자·OS 구분자 통일)
    rels = {
        str(p.relative_to(FEATURE_ROOT / L).with_suffix("")).replace("\\", "/")
        for p in files
    }

    feature_sets[L] = rels
    print(L, len(rels))

# 세 스케일 모두 존재하는 영상만 detection 학습에 사용 가능
common = feature_sets["L8"] & feature_sets["L32"] & feature_sets["L64"]

print("common:", len(common))
print("missing in L8:", len(common ^ feature_sets["L8"]))
print("missing in L32 compared to L8:", len(feature_sets["L8"] - feature_sets["L32"]))
print("missing in L64 compared to L8:", len(feature_sets["L8"] - feature_sets["L64"]))

print("\n=== L32에 없는 파일 예시 (L8에는 있음) ===")
for x in sorted(feature_sets["L8"] - feature_sets["L32"])[:20]:
    print(x)

print("\n=== L64에 없는 파일 예시 (L8에는 있음) ===")
for x in sorted(feature_sets["L8"] - feature_sets["L64"])[:20]:
    print(x)

In [ ]:
from pathlib import Path

CUSTOM_ROOT = Path("C:/4_1_딥러닝_팀플/MTFL_UCF_custom")
FEATURE_ROOT = CUSTOM_ROOT / "features"


def get_feature_set(L):
    """스케일 L 폴더의 feature 상대 경로 집합 반환."""
    files = list((FEATURE_ROOT / L).rglob("*.txt"))
    return {
        str(p.relative_to(FEATURE_ROOT / L).with_suffix("")).replace("\\", "/")
        for p in files
    }


sets = {L: get_feature_set(L) for L in ["L8", "L32", "L64"]}

missing_L32 = sorted(sets["L8"] - sets["L32"])
missing_L64 = sorted(sets["L8"] - sets["L64"])

print("missing L32:", len(missing_L32))
print("missing L64:", len(missing_L64))

# 재추출·수동 점검용 목록 저장
miss_dir = CUSTOM_ROOT / "missing_lists"
miss_dir.mkdir(parents=True, exist_ok=True)

(miss_dir / "missing_L32.txt").write_text("\n".join(missing_L32), encoding="utf-8")
(miss_dir / "missing_L64.txt").write_text("\n".join(missing_L64), encoding="utf-8")

print("saved:", miss_dir / "missing_L32.txt")
print("saved:", miss_dir / "missing_L64.txt")

## 4단계 (정책): 짧은 영상 제외 후 detection 학습

영상 길이 부족으로 L32/L64 feature가 없는 샘플은 **학습·평가에서 제외**합니다.  
아래에서 L8∩L32∩L64 **교집합(common)** 만 남기고 annotation을 `*_common.txt`로 다시 만듭니다.

### L8 ∩ L32 ∩ L64 교집합 크기 확인

In [ ]:
from pathlib import Path

CUSTOM_ROOT = Path("C:/4_1_딥러닝_팀플/MTFL_UCF_custom")
FEATURE_ROOT = CUSTOM_ROOT / "features"
ANNO_ROOT = CUSTOM_ROOT / "annotations"  # 이후 filter_annotation에서 사용


def get_feature_set(L):
    return {
        str(p.relative_to(FEATURE_ROOT / L).with_suffix("")).replace("\\", "/")
        for p in (FEATURE_ROOT / L).rglob("*.txt")
    }


set_L8 = get_feature_set("L8")
set_L32 = get_feature_set("L32")
set_L64 = get_feature_set("L64")

common = set_L8 & set_L32 & set_L64  # detection에 쓸 최종 영상 집합

print("L8:", len(set_L8))
print("L32:", len(set_L32))
print("L64:", len(set_L64))
print("common:", len(common))

### annotation 필터링 → `train_anno_common.txt`, `test_anno_common.txt`

`common`에 포함된 영상만 남깁니다. annotation 첫 컬럼의 `.mp4` 경로를 feature 키 형식으로 변환해 매칭합니다.

In [ ]:
def filter_annotation(input_path, output_path, common):
    """
    train/test annotation에서 feature가 세 스케일 모두 있는 행만 복사.
    common: "Category/video_id" 문자열 집합 (확장자 없음)
    """
    kept = 0
    removed = 0

    with open(input_path, "r", encoding="utf-8") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for line in fin:
            if not line.strip():
                continue

            items = line.strip().split()
            video_path = items[0]  # 예: Shoplifting/Shoplifting007_x264.mp4

            rel = str(Path(video_path).with_suffix("")).replace("\\", "/")

            if rel in common:
                fout.write(line)
                kept += 1
            else:
                removed += 1

    print(output_path.name)
    print(" kept:", kept)
    print(" removed:", removed)


filter_annotation(
    ANNO_ROOT / "train_anno.txt",
    ANNO_ROOT / "train_anno_common.txt",
    common,
)

filter_annotation(
    ANNO_ROOT / "test_anno.txt",
    ANNO_ROOT / "test_anno_common.txt",
    common,
)

### 필터링 후 Normal / Abnormal 비율 확인 (`*_common.txt`)

In [ ]:
from collections import Counter
from pathlib import Path

# 셀 23을 실행하지 않았을 때를 대비해 ANNO_ROOT 재정의
ANNO_ROOT = Path("C:/4_1_딥러닝_팀플/MTFL_UCF_custom/annotations")


def check_label_dist(path):
    """annotation 파일 두 번째 컬럼(label) 분포 출력."""
    labels = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                labels.append(line.split()[1])

    counter = Counter(labels)
    total = len(labels)
    normal = counter.get("Normal", 0)
    abnormal = total - normal

    print("\n", path.name)
    print("total:", total)
    print("normal:", normal)
    print("abnormal:", abnormal)
    print("normal ratio:", normal / total if total else 0)

    for k, v in sorted(counter.items()):
        print(k, v)


check_label_dist(ANNO_ROOT / "train_anno_common.txt")
check_label_dist(ANNO_ROOT / "test_anno_common.txt")

NameError: name 'ANNO_ROOT' is not defined

## 5단계: MTFL detection 모델 학습

- 입력: L64(lf), L32(mf), L8(sf) 멀티스케일 feature + `train_anno_common.txt` / `test_anno_common.txt`
- 출력: `checkpoints_common/` (`.pkl`), `train_results_common/` (`{step}-step-AUC.txt` 등)
- `seg_num=32`: 세그먼트 단위 이상 점수 학습; `feature_size=1024`: Swin feature 차원

In [ ]:
import sys
import subprocess
from pathlib import Path

MTFL_ROOT = "C:/4_1_딥러닝_팀플/MTFL"
CUSTOM_ROOT = "C:/4_1_딥러닝_팀플/MTFL_UCF_custom"

Path(f"{CUSTOM_ROOT}/checkpoints_common").mkdir(parents=True, exist_ok=True)
Path(f"{CUSTOM_ROOT}/train_results_common").mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    f"{MTFL_ROOT}/detection/train.py",

    # L8∩L32∩L64에 맞춘 annotation (feature 누락 샘플 제외)
    "--train_anno", f"{CUSTOM_ROOT}/annotations/train_anno_common.txt",
    "--test_anno", f"{CUSTOM_ROOT}/annotations/test_anno_common.txt",

    "--lf_dir", f"{CUSTOM_ROOT}/features/L64",  # long-term
    "--mf_dir", f"{CUSTOM_ROOT}/features/L32",  # mid-term
    "--sf_dir", f"{CUSTOM_ROOT}/features/L8",    # short-term

    "--save_models", f"{CUSTOM_ROOT}/checkpoints_common",
    "--output_dir", f"{CUSTOM_ROOT}/train_results_common",

    "--gpu", "0",
    "--feature_size", "1024",
    "--seg_num", "32",

    "--batch-size", "64",
    "--workers", "0",   # Windows DataLoader 멀티프로세스 이슈 회피

    "--lr", "0.0001",
    "--max-epoch", "2000",
]

result = subprocess.run(
    cmd,
    cwd=MTFL_ROOT,  # MTFL 패키지 import 기준 디렉터리
)

print("RETURN CODE:", result.returncode)

## 5단계 (확인): 학습 곡선 — step별 AUC / AP

In [ ]:
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt

# train.py가 주기적으로 저장하는 평가 로그 디렉터리
RESULT_DIR = Path("C:/4_1_딥러닝_팀플/MTFL_UCF_custom/train_results_common")

records = []

# 예: 1480-step-AUC.txt → step=1480, 내용 "AUC: 0.85" 형태
for txt_path in RESULT_DIR.glob("*-step-AUC.txt"):
    step_match = re.search(r"(\d+)-step-AUC\.txt", txt_path.name)
    if step_match is None:
        continue

    step = int(step_match.group(1))
    data = {"step": step}

    with open(txt_path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            if ":" not in line:
                continue

            key, value = line.strip().split(":", 1)
            key = key.strip()
            value = value.strip()

            try:
                data[key] = float(value)
            except ValueError:
                pass

    records.append(data)

df = pd.DataFrame(records).sort_values("step")

print(df)

plt.figure(figsize=(8, 5))

if "AUC" in df.columns:
    plt.plot(df["step"], df["AUC"], marker="o", label="AUC")

if "AP" in df.columns:
    plt.plot(df["step"], df["AP"], marker="o", label="AP")

plt.xlabel("Step")
plt.ylabel("Score")
plt.title("MTFL Detection Evaluation Curve")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# AUC만 단독으로 다시 그리기 (가독성용, 위 셀의 df 재사용)
plt.figure(figsize=(8, 5))

if "AUC" in df.columns:
    plt.plot(df["step"], df["AUC"], marker="o", label="AUC")

plt.xlabel("Step")
plt.ylabel("Score")
plt.title("MTFL Detection Evaluation Curve")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from pathlib import Path

RESULT_DIR = Path("C:/4_1_딥러닝_팀플/MTFL_UCF_custom/train_results_common")

print("exists:", RESULT_DIR.exists())

# 저장된 로그·중간 결과 파일명 나열
for p in RESULT_DIR.iterdir():
    print(p.name)

## 5단계 (확인): 저장된 detection checkpoint 목록

In [ ]:
from pathlib import Path

# CUSTOM_ROOT는 학습 셀(29) 실행 후 메모리에 있어야 함; 없으면 경로 문자열로 대체
_custom = globals().get("CUSTOM_ROOT", "C:/4_1_딥러닝_팀플/MTFL_UCF_custom")

ckpts = list(Path(f"{_custom}/checkpoints_common").rglob("*"))
for c in ckpts:
    print(c)  # 예: MTFL-1480.pkl — test.py의 --detection_model에 지정

## 6단계: 학습된 detection 모델로 test set 추론

`detection/test.py`가 test 영상별 **frame-level anomaly score** (`*_scores.npy`)와 GT (`*_gt.npy`)를 `results_eval/scores/`에 저장합니다.

In [ ]:
import sys
import subprocess
from pathlib import Path

MTFL_ROOT = "C:/4_1_딥러닝_팀플/MTFL"
CUSTOM_ROOT = "C:/4_1_딥러닝_팀플/MTFL_UCF_custom"

# ★ 학습에서 가장 성능이 좋았던 step의 pkl로 교체 (예: 1480-step AUC 최고)
detection_model = (
    "C:/4_1_딥러닝_팀플/MTFL_UCF_custom/checkpoints_common/MTFL-1480.pkl"
)

Path(f"{CUSTOM_ROOT}/results_eval").mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    f"{MTFL_ROOT}/detection/test.py",

    "--test_anno", f"{CUSTOM_ROOT}/annotations/test_anno_common.txt",
    "--detection_model", detection_model,

    "--lf_dir", f"{CUSTOM_ROOT}/features/L64",
    "--mf_dir", f"{CUSTOM_ROOT}/features/L32",
    "--sf_dir", f"{CUSTOM_ROOT}/features/L8",

    "--output_dir", f"{CUSTOM_ROOT}/results_eval",
    "--gpu", "0",
    "--feature_size", "1024",
    "--seg_num", "32",
    "--workers", "0",
]

result = subprocess.run(
    cmd,
    cwd=MTFL_ROOT,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

print("RETURN CODE:", result.returncode)
print("\n===== STDOUT =====")
print(result.stdout)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

CUSTOM_ROOT = Path(r"C:/4_1_딥러닝_팀플/MTFL_UCF_custom")
score_root = CUSTOM_ROOT / "results_eval" / "scores"

# test.py 출력: 각 영상마다 {name}_scores.npy, {name}_gt.npy (프레임 단위 0/1 GT)
score_files = sorted(score_root.rglob("*_scores.npy"))

print("score files:", len(score_files))
for p in score_files[:10]:
    print(p)

In [ ]:
# 인덱스로 시각화 대상 영상 선택할 때 사용 (아래 plot_saved_score(score_files[i]))
print("score file 개수:", len(score_files))

for i, p in enumerate(score_files[:20]):
    print(i, p.name)

## 7단계: 이상 점수·GT·대표 프레임 시각화

`MTFL_vis.py` (동일 폴더) 제공 함수:

- `plot_saved_score`: 프레임별 anomaly score + GT 구간 음영
- `plot_score_with_selected_frames`: 점수 곡선 + 지정 프레임 썸네일 (GT 시작·score peak·GT 끝 등)

In [ ]:
# notebooks/동원/MTFL_vis.py — score npy + mp4 프레임 시각화
from MTFL_vis import (
    plot_saved_score,
    plot_score_with_selected_frames,
)

In [ ]:
# 인덱스 11번 test 영상: score 곡선 + GT anomaly 구간
plot_saved_score(score_files[11])

In [ ]:
target_idx = 2  # score_files 중 시각화할 영상 인덱스

score_path = score_files[target_idx]
scores = np.load(score_path)  # shape: (num_frames,) — 0~1 anomaly probability

gt_path = score_path.with_name(
    score_path.name.replace("_scores.npy", "_gt.npy")
)
gt = np.load(gt_path)  # frame-level GT: 1=anomaly, 0=normal

anomaly_idx = np.where(gt == 1)[0]

print("video:", score_path.stem)
print("GT start:", anomaly_idx.min())
print("GT end:", anomaly_idx.max())

# 대표 프레임 3장: GT 구간 시작 / 모델 score 최대 / GT 구간 끝
plot_score_with_selected_frames(
    score_path=score_path,
    video_root=r"C:\4_1_딥러닝_팀플\MTFL_UCF_custom\videos",
    frame_indices=sorted([
        int(anomaly_idx.min()),
        int(np.argmax(scores)),
        int(anomaly_idx.max()),
    ]),
)

In [ ]:
# ---------------------------------------------------------------------------
# (참고/미완성) 다른 데이터셋(AIHUB)용 event hit rate — event_hit_rate 미정의 시 실행 불가
# UCF 파이프라인과 무관하면 이 셀은 생략하세요.
# ---------------------------------------------------------------------------
CUSTOM_ROOT = Path(r"C:\4_1_딥러닝_팀플\MTFL_custom_12_2")
AIHUB_SCORE_ROOT = CUSTOM_ROOT / "results_eval" / "scores"

# details = event_hit_rate(AIHUB_SCORE_ROOT, top_percent=5)
# for d in details[:10]:
#     print(d)